# READ FIRST: Colab runtime (Python) version requirement
**Python 3.13 is incompatible with this project's approved QLoRA environment.** A Colab session that reports `Python 3.13.x` fails Section 3 with
`RuntimeError: Python 3.10-3.12 is required for pinned cloud wheels`. That check is intentional and is **not** bypassed or edited here. This notebook does not (and must not) try to downgrade Python or change packages inside the running session.

**Before running Sections 2-7, select a compatible Colab runtime version** (the approved pins `torch==2.5.1+cu124`, `transformers==4.48.3`, `peft==0.14.0`, `bitsandbytes==0.45.3`, ... require Python 3.10-3.12):
1. `Runtime` -> `Change runtime type`.
2. Hardware accelerator: **T4 GPU**.
3. **Runtime version**: open the dropdown (default `Latest`) and choose an older snapshot that provides **Python 3.12**. Previously verified for this project: **`2026.07`** (Python 3.12.13, T4, PyTorch `2.5.1+cu124` after Section 4, CUDA 12.4). *Whether `2026.07` is still offered today is unconfirmed - use it only if it appears in your dropdown.*
4. `Save`, then `Runtime` -> `Disconnect and delete runtime` once and reconnect (an older image can take a few extra minutes to start). Note that deleting the runtime discards temporary diagnostic evidence under `/content/phase6_diag`.
5. Confirm in a scratch cell: `import sys; print(sys.version)` shows **3.10, 3.11 or 3.12** (3.12.x expected), and `!nvidia-smi` shows a Tesla T4. Then use `Runtime` -> `Run all`.

If no listed runtime version provides Python 3.10-3.12, stop and report it; do not edit the Section 3 check.

| Previously verified configuration | Value |
| :--- | :--- |
| Colab runtime version | `2026.07` (availability not re-verified) |
| Python | 3.12.13 |
| GPU | Tesla T4 (compute capability 7.5) |
| PyTorch / CUDA after Section 4 | `2.5.1+cu124` / 12.4 |
| Failing configuration | Python 3.13.16 (rejected by Section 3, as designed) |


# Phase 6 Colab Preflight (Sections 2-7 only)
Runs the prerequisite sections of `phase6_qlora_training.ipynb` verbatim and **stops after Section 7**. Use **Runtime -> Run all**.

- Colab halts at the first failing cell; do not bypass errors.
- **Section 4** may raise `RESTART REQUIRED` (torch swapped on disk). Then use **Runtime -> Restart session** once and **Run all** again.
- **Section 5** shows a one-time Google Drive authorization prompt (Drive holds `inputs/train.json` and `inputs/validation.json`).
- No base model is downloaded or loaded; no training or GGUF export code is included. Only `train.json` and `validation.json` are used.

| Checkpoint | Meaning |
| :--- | :--- |
| After Section 2 | T4 GPU + FP16 selected |
| After Section 4 | Pinned packages verified |
| After Section 6 | Dataset hashes, counts, ID overlap verified |
| After Section 7 | Compatibility gate passed |


---
### Diagnostics Setup (run first)
Arms a post-run observer so every Section 2-7 cell records: section + heading, UTC timestamp, SUCCEEDED/FAILED status, exception type, message and full traceback, Python/package versions (on disk and in memory), and GPU/CUDA info (`nvidia-smi` and torch when loaded).

- **Before Section 5:** evidence exists only in temporary Colab storage (`/content/phase6_diag/events`). It survives `Restart session` (so the Section 4 `RESTART REQUIRED` evidence is kept) but is lost on `Disconnect and delete runtime`.
- **After Section 5** mounts Drive and verifies it writable, all local evidence is copied (append-only, never overwritten) to `.../slm-customer-support-phase6/diagnostics/`.
- The observer never suppresses an exception, never marks a failed cell successful, and never bypasses a check.


In [ ]:
# ==============================================================================
# PERSISTENT EXECUTION DIAGNOSTICS (run FIRST). Stdlib only: imports no torch/transformers.
# Observes every preflight cell via an IPython post-run hook. It never suppresses,
# alters, retries or re-labels an exception: failed cells stay failed in Colab.
# ==============================================================================
import datetime as _dt
import hashlib as _hl
import importlib.metadata as _im
import json as _json
import os as _os
import platform as _pf
import subprocess as _sp
import sys as _sys
import traceback as _tb
import uuid as _uuid
from pathlib import Path as _P

# sha256(stripped cell source) -> section metadata (generated at build time from the cells below)
_DIAG_SECTION_MAP = {
    "c5a12fdb748f982d7329229097234ff565a6056d44afa129640c0c3c3607688b": {
        "section": "Section 2",
        "heading": "Runtime GPU Verification",
        "cell_id": "sec02-code"
    },
    "14dd787db910a82b8919b196f7c667fb69d75633b4704bf5d7344f6bfa97dfd7": {
        "section": "Section 2",
        "heading": "Runtime GPU Verification",
        "cell_id": "chk02-code"
    },
    "24a19c88efa036f6060908203bb3b6991a83c7366ec6f32ba094a7453ff8eded": {
        "section": "Section 3",
        "heading": "Python, Disk Space, and Optional Secret Checks",
        "cell_id": "sec03-code"
    },
    "444754752a79ef00c557717ba12cfbf8d04b521eb8d5d85bd53138ec9afeeed2": {
        "section": "Section 4",
        "heading": "Cloud Dependency Installation & Two-Pass Runtime Restart Workflow",
        "cell_id": "sec04-code"
    },
    "d63230f47af6060afe26b24e094abd66cef136dfe2b52490546c86d822a50891": {
        "section": "Section 4",
        "heading": "Cloud Dependency Installation & Two-Pass Runtime Restart Workflow",
        "cell_id": "chk04-code"
    },
    "f653769cd4c69bb02b54657bb196fb7e940ff086855b82b31bcb1239a29f6be5": {
        "section": "Section 5",
        "heading": "Secure Google Drive Connection, Write-Permission Probe & Backup Safety Policy",
        "cell_id": "sec05-code"
    },
    "37dc7b2ff42a6112d37c4819282074f278161d8c4b1a5bec73e0f7b1618baec6": {
        "section": "Section 6",
        "heading": "Explicit Upload & SHA-256 Verification of `train.json` and `validation.json` Only",
        "cell_id": "sec06-code"
    },
    "367338f05ae5d96713174853581040cebe07f34cd5f2f1768f82c1c95490f274": {
        "section": "Section 6",
        "heading": "Explicit Upload & SHA-256 Verification of `train.json` and `validation.json` Only",
        "cell_id": "chk06-code"
    },
    "ba85da72759784b8a66f8b1ca849555fac86835ca0f80b160330be695c06d7dd": {
        "section": "Section 7",
        "heading": "Mandatory Pre-Training Runtime Compatibility Gate (Stops Before Model Download on Any Failure)",
        "cell_id": "sec07-code"
    },
    "2ed01b0ef55a271109ad15a9fb03a16d93f7700359d2ad38ee116d413957b0e0": {
        "section": "Final checkpoint",
        "heading": "Preflight complete",
        "cell_id": "pre99-code"
    }
}

# Local evidence survives 'Restart session' (filesystem persists) but is LOST on 'Disconnect and delete runtime'.
_DIAG_LOCAL_DIR = _P("/content/phase6_diag") if _P("/content").exists() else _P("./phase6_diag")
_DIAG_PACKAGES = (
    "torch", "torchvision", "torchaudio", "transformers", "peft", "bitsandbytes",
    "accelerate", "datasets", "safetensors", "huggingface-hub", "pyyaml", "packaging",
)


def _diag_env_snapshot() -> dict:
    snap = {
        "python": _sys.version,
        "platform": _pf.platform(),
        "disk_package_versions": {},
        "in_memory": {},
        "nvidia_smi": None,
    }
    for pkg in _DIAG_PACKAGES:
        try:
            snap["disk_package_versions"][pkg] = _im.version(pkg)
        except Exception as exc:  # package absent or metadata unreadable
            snap["disk_package_versions"][pkg] = f"<unavailable: {type(exc).__name__}>"
    for mod_name in ("torch", "transformers", "peft", "bitsandbytes", "accelerate", "safetensors"):
        mod = _sys.modules.get(mod_name)
        if mod is not None:
            snap["in_memory"][mod_name] = getattr(mod, "__version__", "<no __version__>")
    torch_mod = _sys.modules.get("torch")
    if torch_mod is not None:
        try:
            info = {"cuda_runtime": getattr(torch_mod.version, "cuda", None)}
            info["cuda_available"] = bool(torch_mod.cuda.is_available())
            if info["cuda_available"]:
                props = torch_mod.cuda.get_device_properties(0)
                free_b, total_b = torch_mod.cuda.mem_get_info(0)
                info.update(
                    device_name=props.name,
                    compute_capability=f"{props.major}.{props.minor}",
                    total_vram_gib=round(props.total_memory / 1024 ** 3, 2),
                    free_vram_gib=round(free_b / 1024 ** 3, 2),
                )
            snap["in_memory"]["torch_cuda"] = info
        except Exception as exc:
            snap["in_memory"]["torch_cuda"] = f"<unavailable: {type(exc).__name__}: {exc}>"
    try:
        out = _sp.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free,compute_cap",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=20,
        )
        snap["nvidia_smi"] = (out.stdout or out.stderr).strip()
    except Exception as exc:
        snap["nvidia_smi"] = f"<unavailable: {type(exc).__name__}>"
    return snap


def _diag_drive_dir():
    """Return the Drive diagnostics folder ONLY after Section 5 mounted Drive and verified it writable."""
    g = globals()
    if g.get("DRIVE_ROOT_DIR") is None or g.get("DRIVE_WRITE_PROBE_INFO") is None:
        return None
    if not _os.path.ismount("/content/drive"):
        return None
    return _P(g["DRIVE_ROOT_DIR"]) / "diagnostics"


def _diag_sync_to_drive() -> dict:
    """Copy local evidence to Drive. Append-only: unique filenames, existing files are never overwritten."""
    drive_dir = _diag_drive_dir()
    if drive_dir is None:
        return {"drive": "NOT_AVAILABLE", "copied": 0, "skipped_existing": 0, "error": None}
    result = {"drive": str(drive_dir), "copied": 0, "skipped_existing": 0, "error": None}
    try:
        drive_dir.mkdir(parents=True, exist_ok=True)
        for src in sorted((_DIAG_LOCAL_DIR / "events").glob("*.json")):
            dst = drive_dir / src.name
            if dst.exists():
                result["skipped_existing"] += 1
                continue
            tmp = drive_dir / f".{src.name}.{_uuid.uuid4().hex}.part"
            try:
                data = src.read_bytes()
                with open(tmp, "xb") as fh:
                    fh.write(data)
                    fh.flush()
                    _os.fsync(fh.fileno())
                _os.replace(tmp, dst)
            finally:
                if tmp.exists():
                    tmp.unlink()
            if _hl.sha256(dst.read_bytes()).hexdigest() != _hl.sha256(data).hexdigest():
                raise IOError(f"Drive diagnostic copy SHA-256 mismatch: {dst}")
            result["copied"] += 1
    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"
    return result


def _diag_record(entry: dict) -> None:
    events_dir = _DIAG_LOCAL_DIR / "events"
    events_dir.mkdir(parents=True, exist_ok=True)
    stamp = _dt.datetime.now(_dt.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    slug = entry["section"].replace(" ", "_").replace("/", "-")
    path = events_dir / f"{stamp}_{slug}_{entry['status']}_{_uuid.uuid4().hex[:8]}.json"
    tmp = path.with_suffix(".part")
    tmp.write_text(_json.dumps(entry, indent=2, default=str), encoding="utf-8")
    _os.replace(tmp, path)
    sync = _diag_sync_to_drive()
    if sync["drive"] == "NOT_AVAILABLE":
        where = f"LOCAL TEMPORARY COLAB STORAGE ONLY ({events_dir}); Drive not yet mounted/verified writable"
    elif sync["error"]:
        where = f"LOCAL ONLY ({events_dir}); Drive sync FAILED: {sync['error']}"
    else:
        where = f"local {events_dir} + Drive {sync['drive']} (copied {sync['copied']} new)"
    print(f"[DIAG] {entry['section']} -> {entry['status']} | evidence: {where}")


def _diag_post_run_cell(result) -> None:
    try:
        raw = getattr(getattr(result, "info", None), "raw_cell", None)
        if not raw:
            return
        meta = _DIAG_SECTION_MAP.get(_hl.sha256(raw.strip().encode("utf-8")).hexdigest())
        if meta is None:
            return
        err = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
        entry = {
            "section": meta["section"],
            "heading": meta["heading"],
            "cell_id": meta["cell_id"],
            "timestamp_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
            "status": "FAILED" if err is not None else "SUCCEEDED",
            "exception_type": None,
            "exception_message": None,
            "traceback": None,
            "restart_required_condition": False,
            "environment": _diag_env_snapshot(),
        }
        if err is not None:
            entry["exception_type"] = f"{type(err).__module__}.{type(err).__qualname__}"
            entry["exception_message"] = str(err)
            entry["traceback"] = "".join(_tb.format_exception(type(err), err, err.__traceback__))
            entry["restart_required_condition"] = "RESTART REQUIRED" in str(err)
        _diag_record(entry)
    except Exception as hook_exc:  # diagnostics must never alter notebook behaviour
        print(f"[DIAG] diagnostics hook error (original cell result is unaffected): {hook_exc!r}")


_ip = get_ipython()
_prev = globals().get("_DIAG_REGISTERED_HOOK")
if _prev is not None:
    try:
        _ip.events.unregister("post_run_cell", _prev)
    except ValueError:
        pass
_ip.events.register("post_run_cell", _diag_post_run_cell)
_DIAG_REGISTERED_HOOK = _diag_post_run_cell

print("Diagnostics armed for Sections 2-7 (observer only; errors are never suppressed).")
print(f"  - Local evidence folder : {_DIAG_LOCAL_DIR / 'events'}  (TEMPORARY Colab storage)")
print("      survives 'Restart session' (incl. the Section 4 restart); LOST on 'Disconnect and delete runtime'.")
print("  - Drive copy            : starts automatically after Section 5 mounts Drive and verifies it writable.")
print("      Everything from Sections 2-4 stays local-only until then.")


---
### 2. Runtime GPU Verification
Verifies CUDA availability, checks minimum VRAM (`>= 8 GB`) and compute capability (`>= 7.0`), pins visibility to GPU `0`, enforces **FP16** on NVIDIA T4 (`sm_75`, explicitly rejecting `BF16`), and dynamically selects **BF16** only on Ampere+ GPUs (`sm_80+`) with hardware `bfloat16` support.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

MIN_VRAM_GB = 8.0
MIN_COMPUTE_CAPABILITY = 7.0

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected! In Google Colab, select Runtime -> Change runtime type -> T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
gpu_vram_gb = props.total_memory / (1024 ** 3)
free_vram_bytes, total_vram_bytes = torch.cuda.mem_get_info(0)
free_vram_gb = free_vram_bytes / (1024 ** 3)
cc_tuple = (props.major, props.minor)
cc_float = float(f"{props.major}.{props.minor}")
bf16_hw_supported = bool(torch.cuda.is_bf16_supported())
IS_T4_GPU = ("T4" in gpu_name.upper()) or (cc_tuple == (7, 5))

if cc_float < MIN_COMPUTE_CAPABILITY:
    raise RuntimeError(f"Unsupported GPU {gpu_name} (cc={cc_float} < {MIN_COMPUTE_CAPABILITY}).")
if gpu_vram_gb < MIN_VRAM_GB:
    raise RuntimeError(f"Insufficient total GPU VRAM: {gpu_vram_gb:.2f} GB < {MIN_VRAM_GB:.2f} GB.")

# Dynamic precision policy: NVIDIA T4 (sm_75) strictly uses FP16; Ampere+ (sm_80+) uses BF16
USE_BF16 = bool((not IS_T4_GPU) and props.major >= 8 and bf16_hw_supported)
USE_FP16 = not USE_BF16
assert USE_FP16 != USE_BF16, "Safety error: Exactly one of FP16 or BF16 must be active."
if IS_T4_GPU and (USE_BF16 or not USE_FP16):
    raise RuntimeError("NVIDIA T4 requires FP16 training; BF16 configuration is rejected.")

COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
PRECISION_LABEL = "bfloat16" if USE_BF16 else "float16"

print(f"GPU Verified       : {gpu_name} (is_t4={IS_T4_GPU})")
print(f"Total / Free VRAM  : {gpu_vram_gb:.2f} GiB total / {free_vram_gb:.2f} GiB free")
print(f"Compute Capability : {cc_float}")
print(f"Selected Precision : {PRECISION_LABEL} (fp16={USE_FP16}, bf16={USE_BF16})")

In [ ]:
print('CHECKPOINT: Section 2 completed successfully.')


---
### 3. Python, Disk Space, and Optional Secret Checks
Checks Python version (`3.10–3.12`), OS platform, available ephemeral disk space (`>= 15 GiB` recommended for clean FP16 merging and GGUF export), and optional `HF_TOKEN` from Colab Secrets / environment variables (never hardcoded).

In [ ]:
import platform
import shutil
import sys
from pathlib import Path

py_ver = sys.version_info
if not ((3, 10) <= (py_ver.major, py_ver.minor) <= (3, 12)):
    raise RuntimeError(f"Python 3.10–3.12 is required for pinned cloud wheels; detected {sys.version}")

disk = shutil.disk_usage("/content" if Path("/content").exists() else ".")
free_disk_gib = disk.free / (1024 ** 3)
print(f"Python Version     : {sys.version.split()[0]} ({platform.platform()})")
print(f"PyTorch / CUDA     : {torch.__version__} / CUDA {torch.version.cuda}")
print(f"Free Disk Space    : {free_disk_gib:.2f} GiB")

if free_disk_gib < 15.0:
    raise RuntimeError(f"At least 15 GiB of free disk space is recommended; found {free_disk_gib:.2f} GiB.")

# Optional Hugging Face token via Colab Secrets or environment variable (NEVER hardcode tokens)
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
print(f"HF_TOKEN Status    : {'Loaded from secure secret/env' if HF_TOKEN else 'Not set (public anonymous access)'}")

---
### 4. Cloud Dependency Installation & Two-Pass Runtime Restart Workflow
Pins the exact CUDA 12.4 PyTorch stack (`torch==2.5.1+cu124`, `torchvision==0.20.1+cu124`, `torchaudio==2.5.1+cu124`) from PyTorch's official wheel index alongside the exact Hugging Face QLoRA stack matching `requirements-cloud.txt` (`transformers==4.48.3`, `peft==0.14.0`, `bitsandbytes==0.45.3`, `datasets==3.2.0`, `accelerate==1.3.0`, `safetensors==0.5.2`, `huggingface-hub==0.28.1`, `pyyaml==6.0.2`).

> **EXPECTED TWO-PASS WORKFLOW ON GOOGLE COLAB (WHEN COLAB PRE-INSTALLS PYTORCH >= 2.8):**
> - **Pass 1 (Initial Run):** Replaces Colab's pre-installed PyTorch on disk with `torch==2.5.1+cu124`, verifies the disk install in a fresh subprocess, and **intentionally stops** with a `RESTART REQUIRED` prompt because Section 2 already loaded the older/newer pre-installed `torch` C++ binaries into kernel RAM.
> - **Restart Action:** Click **`Runtime -> Restart session`** *(do NOT click "Disconnect and delete runtime")*. Restarting the session clears Python RAM variables (`USE_FP16`, `COMPUTE_DTYPE`, `WORK_ROOT`, `HF_TOKEN`) while keeping all installed wheels on disk.
> - **Pass 2 (After Restart):** Re-run **Section 2 → Section 3 → Section 4** in order. Section 4 will detect `torch==2.5.1+cu124` active in memory, skip re-downloading wheels (`< 1s`), and verify all imports cleanly. Then continue to **Sections 5, 6, and 7** before model download.

In [ ]:
import importlib.metadata
import subprocess
import sys

TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu124"
TARGET_TORCH_VERSION = "2.5.1+cu124"

TORCH_PACKAGES = [
    "torch==2.5.1+cu124",
    "torchvision==0.20.1+cu124",
    "torchaudio==2.5.1+cu124",
]

CLOUD_PACKAGES = {
    "transformers": "4.48.3",
    "peft": "0.14.0",
    "bitsandbytes": "0.45.3",
    "datasets": "3.2.0",
    "accelerate": "1.3.0",
    "safetensors": "0.5.2",
    "huggingface-hub": "0.28.1",
    "pyyaml": "6.0.2",
}


def _installed_ver(dist_name: str) -> str | None:
    try:
        return importlib.metadata.version(dist_name)
    except importlib.metadata.PackageNotFoundError:
        return None


loaded_torch = sys.modules.get("torch")
active_torch_ver = getattr(loaded_torch, "__version__", None)
disk_torch_ok = (
    _installed_ver("torch") == TARGET_TORCH_VERSION
    and _installed_ver("torchvision") == "0.20.1+cu124"
    and _installed_ver("torchaudio") == "2.5.1+cu124"
)
disk_cloud_ok = all(_installed_ver(pkg) == ver for pkg, ver in CLOUD_PACKAGES.items())

if not disk_torch_ok:
    print(f"Installing reproducible PyTorch CUDA 12.4 stack ({TARGET_TORCH_VERSION})...")
    # Remove optional pre-installed xformers if present to avoid C++/CUDA ABI conflicts
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "--quiet", "xformers"],
        check=False,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--index-url",
            TORCH_INDEX_URL,
            *TORCH_PACKAGES,
        ],
        check=True,
    )

if not disk_cloud_ok:
    cloud_specs = [f"{pkg}=={ver}" for pkg, ver in CLOUD_PACKAGES.items()]
    print("Installing pinned Hugging Face QLoRA cloud dependencies...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *cloud_specs],
        check=True,
    )

# Verify on-disk environment in an isolated subprocess (unaffected by stale sys.modules)
subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch, torchvision, bitsandbytes, transformers, peft, accelerate, datasets, safetensors; "
            f"assert torch.__version__ == '{TARGET_TORCH_VERSION}', torch.__version__; "
            "assert transformers.__version__ == '4.48.3', transformers.__version__; "
            "assert peft.__version__ == '0.14.0', peft.__version__; "
            "assert bitsandbytes.__version__ == '0.45.3', bitsandbytes.__version__"
        ),
    ],
    check=True,
)

# Enforce kernel restart if torch was absent or loaded in RAM with a different version (e.g., 2.11.0+cu128)
if active_torch_ver != TARGET_TORCH_VERSION:
    raise RuntimeError(
        f"RESTART REQUIRED: Disk environment is now verified at torch=={TARGET_TORCH_VERSION}, "
        f"but the active Python kernel has torch={active_torch_ver!r} in memory. "
        "Select 'Runtime -> Restart session' in Google Colab NOW (do NOT delete runtime), "
        "then re-run Sections 2, 3, 4, 5, 6, and 7 in order."
    )

import torch
import accelerate
import bitsandbytes as bnb
import datasets
import huggingface_hub
import peft
import safetensors
import transformers
import yaml

print("Verified Active Cloud Package Versions (In-Memory == On-Disk):")
for pkg_name, mod in [
    ("torch", torch),
    ("transformers", transformers),
    ("peft", peft),
    ("bitsandbytes", bnb),
    ("accelerate", accelerate),
    ("datasets", datasets),
    ("safetensors", safetensors),
    ("huggingface_hub", huggingface_hub),
    ("pyyaml", yaml),
]:
    print(f"  - {pkg_name:16s}: {getattr(mod, '__version__', 'installed')}")

In [ ]:
print('CHECKPOINT: Section 4 completed successfully.')


---
### 5. Secure Google Drive Connection, Write-Permission Probe & Backup Safety Policy
Set `MOUNT_GOOGLE_DRIVE = True` (recommended) to persist checkpoints, training logs, the PEFT LoRA adapter, and the final `~1.04 GiB` quantized GGUF model to Google Drive and optionally load `train.json` and `validation.json` from exact paths after runtime restarts.

**Google Drive Backup Safety Controls (`W-02`–`W-06` Hardened):**
- **`MOUNT_GOOGLE_DRIVE = True`**: Mounts Google Drive by default so the persistent `inputs/train.json` and `inputs/validation.json` are loaded automatically in Section 6 (no re-upload). Colab shows a one-time Google authorization prompt. Set `False` only for ephemeral-only runs (then Section 6 falls back to the allowlisted manual upload).
- **`REQUIRE_GOOGLE_DRIVE_BACKUP = True`**: When `True`, post-training backup and GGUF export stages (`Sections 16, 17, 20–22`) halt immediately with a descriptive error if Google Drive is unmounted, disconnected, or unwritable.
- **`ALLOW_OVERWRITE_DRIVE_BACKUPS = False`**: Separate explicit authorization required before overwriting any existing adapter, checkpoint, training log, or GGUF backup on Google Drive.
- **`BACKUP_CHECKPOINTS_TO_DRIVE = True`**: Explicitly backs up saved `Trainer` checkpoints (`checkpoint-*`, including `best_model_checkpoint`) and a SHA-256 checkpoint manifest to `exports/checkpoints/`.
- **Safe Temporary Write-Probe (`verify_drive_directory_writable`)**: Verifies live Google Drive mount status, writes/`fsync`s/reads back a uniquely named `.colab_drive_write_probe_<pid>_<uuid>.tmp` file, checks free quota, and removes **only** that temporary file.

**Google Drive Folder Structure (`MOUNT_GOOGLE_DRIVE = True`):**
```text
/content/drive/MyDrive/slm-customer-support-phase6/
├── inputs/                                          # Optional exact input location (NO wildcards used)
│   ├── train.json                                   # 250 training records (SHA-256 verified)
│   └── validation.json                              # 50 validation records (SHA-256 verified)
└── exports/                                         # Verified non-destructive output backup location
    ├── customer-support-lora/                       # Saved PEFT LoRA adapter + tokenizer (dual SHA-256 verified)
    ├── checkpoints/                                 # Saved Trainer step/best checkpoints + SHA-256 manifest
    ├── training_log.json                            # Post-training provenance log (dual SHA-256 verified)
    ├── training_log_post_export.json                # Post-GGUF provenance log when overwrite lock is active
    └── qwen2.5-1.5b-customer-support-q4_k_m.gguf    # Final quantized model (dual SHA-256 verified)
```

In [ ]:
import hashlib
import json
import os
import shutil
import uuid
from pathlib import Path

# ==============================================================================
# GOOGLE DRIVE PERSISTENCE & BACKUP SAFETY CONTROLS
# ==============================================================================
MOUNT_GOOGLE_DRIVE = True             # Mounts Google Drive (required for persistent inputs/ and verified backups); Colab shows a one-time authorization prompt
REQUIRE_GOOGLE_DRIVE_BACKUP = True     # Stop backup/export stages if Drive is unmounted, unavailable, or unwritable
ALLOW_OVERWRITE_DRIVE_BACKUPS = False  # Separate explicit authorization required to overwrite existing Drive backups
BACKUP_CHECKPOINTS_TO_DRIVE = True     # Preserve saved Trainer checkpoints to exports/checkpoints/ on Google Drive

WORK_ROOT = Path("/content/slm-phase6-workspace") if Path("/content").exists() else Path("./colab_workspace")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT_DIR = None
DRIVE_INPUTS_DIR = None
DRIVE_EXPORT_DIR = None
DRIVE_WRITE_PROBE_INFO = None


def compute_file_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute SHA-256 checksum by reading a file from disk in binary chunks."""
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            chunk = fh.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def verify_drive_directory_writable(target_dir: Path, min_free_bytes: int = 0) -> dict:
    """Verify directory writability via a uniquely named temporary probe file and remove ONLY that probe file."""
    target_dir = Path(target_dir)
    normalized_target = str(target_dir).replace("\\", "/")
    if normalized_target.startswith("/content/drive") and not os.path.ismount("/content/drive"):
        raise RuntimeError(
            "Google Drive mountpoint '/content/drive' is not an active filesystem mount "
            "(Drive is unmounted or disconnected). Set MOUNT_GOOGLE_DRIVE = True and re-run Section 5."
        )
    if not target_dir.exists() or not target_dir.is_dir():
        raise RuntimeError(f"Google Drive backup directory does not exist or is not a directory: {target_dir}")

    probe_name = f".colab_drive_write_probe_{os.getpid()}_{uuid.uuid4().hex}.tmp"
    probe_path = target_dir / probe_name
    if probe_path.exists():
        raise FileExistsError(f"Refusing write probe because temporary filename unexpectedly exists: {probe_path}")

    expected_payload = f"slm_phase6_drive_write_probe:{uuid.uuid4().hex}\n".encode("utf-8")
    try:
        with open(probe_path, "xb") as probe_fh:
            probe_fh.write(expected_payload)
            probe_fh.flush()
            os.fsync(probe_fh.fileno())

        read_back = probe_path.read_bytes()
        if read_back != expected_payload:
            raise IOError(
                f"Google Drive write-probe read-back mismatch in '{target_dir}': "
                f"wrote {len(expected_payload)} bytes, read {len(read_back)} bytes."
            )

        usage = shutil.disk_usage(target_dir)
        if min_free_bytes > 0 and usage.free < min_free_bytes:
            raise RuntimeError(
                f"Insufficient free space in Google Drive backup directory '{target_dir}': "
                f"available={usage.free / (1024 ** 3):.2f} GiB, required={min_free_bytes / (1024 ** 3):.2f} GiB."
            )
    except Exception as exc:
        raise RuntimeError(
            f"Google Drive directory '{target_dir}' is unavailable or unwritable: {exc}"
        ) from exc
    finally:
        # Remove strictly and ONLY the uniquely named temporary probe file created above
        if (
            (probe_path.is_symlink() or probe_path.is_file())
            and probe_path.parent == target_dir
            and probe_path.name == probe_name
        ):
            probe_path.unlink()

    if probe_path.exists():
        raise RuntimeError(f"Temporary write-probe file could not be removed cleanly: {probe_path}")

    return {
        "target_dir": str(target_dir),
        "probe_filename": probe_name,
        "free_bytes": usage.free,
        "free_gib": round(usage.free / (1024 ** 3), 2),
        "writable": True,
    }


def ensure_drive_backup_ready(stage_name: str, min_free_bytes: int = 0) -> Path | None:
    """Ensure Google Drive is mounted and writable when required; halt clearly if unavailable/unwritable."""
    if not MOUNT_GOOGLE_DRIVE or DRIVE_EXPORT_DIR is None:
        if REQUIRE_GOOGLE_DRIVE_BACKUP:
            raise RuntimeError(
                f"[{stage_name}] Google Drive backup is required (REQUIRE_GOOGLE_DRIVE_BACKUP=True), "
                "but Google Drive is not mounted (MOUNT_GOOGLE_DRIVE=False or DRIVE_EXPORT_DIR is None). "
                "To preserve artifacts safely, set MOUNT_GOOGLE_DRIVE = True in Section 5, run Section 5 to verify "
                "Drive write access, and then re-run this cell (or explicitly set REQUIRE_GOOGLE_DRIVE_BACKUP = False "
                "if you intentionally want ephemeral-only local Colab output)."
            )
        print(
            f"WARNING [{stage_name}]: Google Drive backup skipped because MOUNT_GOOGLE_DRIVE=False "
            "and REQUIRE_GOOGLE_DRIVE_BACKUP=False. Artifacts exist ONLY on ephemeral Colab storage!"
        )
        return None

    probe_result = verify_drive_directory_writable(DRIVE_EXPORT_DIR, min_free_bytes=min_free_bytes)
    print(
        f"[{stage_name}] Verified Google Drive export directory writable: {DRIVE_EXPORT_DIR} "
        f"(free={probe_result['free_gib']:.2f} GiB)"
    )
    return DRIVE_EXPORT_DIR


def safe_backup_file_to_drive(src_path: Path, dst_path: Path, allow_overwrite: bool = False) -> dict:
    """Atomically copy a file to Google Drive with overwrite protection and dual source/destination SHA-256 verification."""
    src_path = Path(src_path)
    dst_path = Path(dst_path)
    if not src_path.is_file():
        raise FileNotFoundError(f"Backup source file does not exist: {src_path}")
    if src_path.resolve() == dst_path.resolve():
        raise ValueError(f"Source and destination backup paths must be distinct: {src_path}")
    if dst_path.exists() and dst_path.is_dir():
        raise IsADirectoryError(f"Destination file path is an existing directory: {dst_path}")
    if dst_path.exists() and not allow_overwrite:
        raise FileExistsError(
            f"Existing Google Drive backup detected at '{dst_path}'. "
            "Overwriting existing backups is blocked by default; set ALLOW_OVERWRITE_DRIVE_BACKUPS = True "
            "in Section 5 only if you intentionally authorize replacing existing Drive backups."
        )

    src_size = src_path.stat().st_size
    src_sha256 = compute_file_sha256(src_path)

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    staged_name = f".{dst_path.name}.{os.getpid()}.{uuid.uuid4().hex}.part"
    staged_path = dst_path.parent / staged_name
    try:
        with open(src_path, "rb") as src_fh, open(staged_path, "xb") as dst_fh:
            while True:
                chunk = src_fh.read(1024 * 1024)
                if not chunk:
                    break
                dst_fh.write(chunk)
            dst_fh.flush()
            os.fsync(dst_fh.fileno())
        try:
            shutil.copystat(src_path, staged_path)
        except OSError:
            pass

        if dst_path.exists() and not allow_overwrite:
            raise FileExistsError(
                f"Existing Google Drive backup detected at '{dst_path}'; "
                "refusing overwrite without ALLOW_OVERWRITE_DRIVE_BACKUPS = True."
            )
        os.replace(staged_path, dst_path)
    except Exception:
        if (
            (staged_path.is_symlink() or staged_path.is_file())
            and staged_path.parent == dst_path.parent
            and staged_path.name == staged_name
        ):
            staged_path.unlink()
        raise

    dst_size = dst_path.stat().st_size
    dst_sha256 = compute_file_sha256(dst_path)
    if src_size != dst_size or src_sha256 != dst_sha256:
        raise IOError(
            f"Google Drive backup SHA-256/size verification FAILED for '{dst_path}': "
            f"source=({src_size} bytes, sha256={src_sha256}) vs "
            f"destination=({dst_size} bytes, sha256={dst_sha256})."
        )

    return {
        "source": str(src_path),
        "destination": str(dst_path),
        "size_bytes": dst_size,
        "source_sha256": src_sha256,
        "destination_sha256": dst_sha256,
        "verified": True,
    }


def safe_backup_directory_to_drive(src_dir: Path, dst_dir: Path, allow_overwrite: bool = False) -> dict:
    """Copy a directory tree to Google Drive with overwrite protection and per-file dual SHA-256 verification."""
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    if not src_dir.is_dir():
        raise FileNotFoundError(f"Backup source directory does not exist: {src_dir}")
    if dst_dir.exists() and not dst_dir.is_dir():
        raise FileExistsError(f"Destination directory path exists as a non-directory: {dst_dir}")

    rel_files = sorted(p.relative_to(src_dir) for p in src_dir.rglob("*") if p.is_file())
    if not rel_files:
        raise RuntimeError(f"Backup source directory contains zero files: {src_dir}")

    if dst_dir.exists() and not allow_overwrite:
        raise FileExistsError(
            f"Existing Google Drive backup directory detected at '{dst_dir}'. "
            "Overwriting existing backups is blocked by default; set ALLOW_OVERWRITE_DRIVE_BACKUPS = True "
            "in Section 5 only if you intentionally authorize replacing existing Drive backups."
        )

    verified_files = []
    total_bytes = 0
    for rel_path in rel_files:
        record = safe_backup_file_to_drive(
            src_dir / rel_path,
            dst_dir / rel_path,
            allow_overwrite=allow_overwrite,
        )
        verified_files.append(
            {
                "relative_path": str(rel_path).replace("\\", "/"),
                "size_bytes": record["size_bytes"],
                "source_sha256": record["source_sha256"],
                "destination_sha256": record["destination_sha256"],
            }
        )
        total_bytes += record["size_bytes"]

    return {
        "source_dir": str(src_dir),
        "destination_dir": str(dst_dir),
        "files_verified": len(verified_files),
        "total_bytes": total_bytes,
        "files": verified_files,
    }


def backup_training_checkpoints_to_drive(
    checkpoint_root: Path,
    drive_export_dir: Path,
    best_checkpoint_path: str | None = None,
    allow_overwrite: bool = False,
) -> dict:
    """Explicitly back up saved Trainer checkpoints (including best_model_checkpoint) and a SHA-256 manifest to Drive."""
    checkpoint_root = Path(checkpoint_root)
    drive_export_dir = Path(drive_export_dir)
    if not checkpoint_root.is_dir():
        raise FileNotFoundError(f"Checkpoint directory does not exist: {checkpoint_root}")

    def _ckpt_sort_key(path: Path) -> tuple[int, int, str]:
        suffix = path.name.rsplit("-", 1)[-1]
        return (0, int(suffix), path.name) if suffix.isdigit() else (1, 0, path.name)

    ckpt_dirs = sorted(
        (p for p in checkpoint_root.glob("checkpoint-*") if p.is_dir()),
        key=_ckpt_sort_key,
    )
    best_ckpt_resolved = Path(best_checkpoint_path).resolve() if best_checkpoint_path else None
    if best_ckpt_resolved is not None and best_ckpt_resolved.is_dir():
        if all(p.resolve() != best_ckpt_resolved for p in ckpt_dirs):
            ckpt_dirs.insert(0, Path(best_checkpoint_path))

    if not ckpt_dirs:
        raise FileNotFoundError(f"No saved Trainer checkpoints ('checkpoint-*') found in: {checkpoint_root}")

    drive_ckpt_root = drive_export_dir / "checkpoints"
    drive_manifest_path = drive_ckpt_root / "checkpoint_backup_manifest.json"
    if not allow_overwrite:
        if drive_manifest_path.exists():
            raise FileExistsError(
                f"Existing checkpoint backup manifest detected at '{drive_manifest_path}'. "
                "Set ALLOW_OVERWRITE_DRIVE_BACKUPS = True in Section 5 to replace existing checkpoint backups."
            )
        for ckpt_dir in ckpt_dirs:
            dst_ckpt_dir = drive_ckpt_root / ckpt_dir.name
            if dst_ckpt_dir.exists():
                raise FileExistsError(
                    f"Existing checkpoint backup detected at '{dst_ckpt_dir}'. "
                    "Set ALLOW_OVERWRITE_DRIVE_BACKUPS = True in Section 5 to replace existing checkpoint backups."
                )

    backed_up_entries = []
    total_files = 0
    total_bytes = 0
    for ckpt_dir in ckpt_dirs:
        summary = safe_backup_directory_to_drive(
            ckpt_dir,
            drive_ckpt_root / ckpt_dir.name,
            allow_overwrite=allow_overwrite,
        )
        is_best = best_ckpt_resolved is not None and ckpt_dir.resolve() == best_ckpt_resolved
        backed_up_entries.append(
            {
                "checkpoint_name": ckpt_dir.name,
                "is_best_model_checkpoint": is_best,
                "files_verified": summary["files_verified"],
                "total_bytes": summary["total_bytes"],
                "destination_dir": summary["destination_dir"],
                "files": summary["files"],
            }
        )
        total_files += summary["files_verified"]
        total_bytes += summary["total_bytes"]

    manifest_payload = {
        "best_model_checkpoint": Path(best_checkpoint_path).name if best_checkpoint_path else None,
        "checkpoint_count": len(backed_up_entries),
        "total_files_verified": total_files,
        "total_bytes": total_bytes,
        "checkpoints": backed_up_entries,
    }
    local_manifest_path = checkpoint_root / "checkpoint_backup_manifest.json"
    local_manifest_path.write_text(json.dumps(manifest_payload, indent=2), encoding="utf-8")
    manifest_backup = safe_backup_file_to_drive(
        local_manifest_path,
        drive_manifest_path,
        allow_overwrite=allow_overwrite,
    )

    return {
        "best_model_checkpoint": manifest_payload["best_model_checkpoint"],
        "checkpoint_names": [entry["checkpoint_name"] for entry in backed_up_entries],
        "checkpoint_count": len(backed_up_entries),
        "total_files_verified": total_files,
        "total_bytes": total_bytes,
        "destination_root": str(drive_ckpt_root),
        "manifest_destination": manifest_backup["destination"],
        "manifest_sha256": manifest_backup["destination_sha256"],
    }


if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT_DIR = Path("/content/drive/MyDrive/slm-customer-support-phase6")
    DRIVE_INPUTS_DIR = DRIVE_ROOT_DIR / "inputs"
    DRIVE_EXPORT_DIR = DRIVE_ROOT_DIR / "exports"
    DRIVE_INPUTS_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_WRITE_PROBE_INFO = verify_drive_directory_writable(DRIVE_EXPORT_DIR)
    print("Google Drive mounted and verified writable via temporary write-probe.")
    print(f"  - Optional exact input folder : {DRIVE_INPUTS_DIR}")
    print(f"  - Output export backup folder : {DRIVE_EXPORT_DIR} (free={DRIVE_WRITE_PROBE_INFO['free_gib']:.2f} GiB)")
    print(f"  - Require Drive backup        : {REQUIRE_GOOGLE_DRIVE_BACKUP}")
    print(f"  - Overwrite existing backups  : {ALLOW_OVERWRITE_DRIVE_BACKUPS}")
    print(f"  - Backup checkpoints to Drive : {BACKUP_CHECKPOINTS_TO_DRIVE}")
else:
    print(f"Using ephemeral Colab workspace directory: {WORK_ROOT.resolve()}")
    if REQUIRE_GOOGLE_DRIVE_BACKUP:
        print(
            "NOTICE: MOUNT_GOOGLE_DRIVE is False while REQUIRE_GOOGLE_DRIVE_BACKUP is True.\n"
            "  - Set MOUNT_GOOGLE_DRIVE = True and re-run Section 5 before running post-training backup/export cells "
            "(Sections 16, 17, 20–22), or those stages will halt safely rather than silently skipping Drive backup."
        )
    else:
        print("WARNING: Both MOUNT_GOOGLE_DRIVE and REQUIRE_GOOGLE_DRIVE_BACKUP are False (ephemeral-only mode).")

---
### 6. Explicit Upload & SHA-256 Verification of `train.json` and `validation.json` Only
Loads **ONLY** `train.json` (`250` records) and `validation.json` (`50` records) without any wildcard file discovery:

1. **If Google Drive is mounted (`MOUNT_GOOGLE_DRIVE = True`, the default)**, both exact files MUST exist at `/content/drive/MyDrive/slm-customer-support-phase6/inputs/train.json` and `/content/drive/MyDrive/slm-customer-support-phase6/inputs/validation.json`. They are SHA-256 verified on Drive, copied into the workspace, and verified again. **No upload is requested.** If either file is missing or has a wrong hash, the cell stops with a clear error; no other dataset is substituted.
2. **Only if Drive is not mounted (`MOUNT_GOOGLE_DRIVE = False`), an interactive upload prompt opens:**
   - Navigate to `C:\slm-customer-support\data\splits\` on your local machine.
   - Hold **`Ctrl` + Click** (Windows/Linux) or **`Cmd` + Click** (macOS) to select **both `train.json` and `validation.json` together** in one upload dialog, then click **Open**.
   - **Never upload the full project directory and never select any other file.** Any unexpected filename (including browser duplicates such as `train (1).json`) is immediately deleted and rejected.
3. Both files are verified against their **exact expected record counts (`250` and `50`) and recorded project SHA-256 hashes**.

In [ ]:
import hashlib
import json

DATA_DIR = WORK_ROOT / "data_splits"
DATA_DIR.mkdir(parents=True, exist_ok=True)

ALLOWED_UPLOAD_FILENAMES = {"train.json", "validation.json"}
EXPECTED_SPLIT_SPECS = {
    "train.json": {
        "count": 250,
        "sha256": "8100ffc6a8d064bd18d20f84562134fc76f554798e813eaa3d2ff87826686cce",
    },
    "validation.json": {
        "count": 50,
        "sha256": "113cd840eae1198db032b1fec6374f3f81f7ec43fab0c44034e8ebd96cfcee9d",
    },
}
REQUIRED_RECORD_KEYS = {
    "id",
    "category",
    "question",
    "context",
    "expected_answer",
    "requires_escalation",
    "difficulty",
}
FORBIDDEN_CHATML_TOKENS = ("<|endoftext|>", "<|im_start|>", "<|im_end|>")


def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


train_dst = DATA_DIR / "train.json"
val_dst = DATA_DIR / "validation.json"

# Option A (Drive mounted): load ONLY the exact persistent files from Drive inputs/ (no wildcards, no upload, no substitution)
if DRIVE_INPUTS_DIR is not None:
    for split_name in sorted(ALLOWED_UPLOAD_FILENAMES):
        drive_file = DRIVE_INPUTS_DIR / split_name
        if not drive_file.is_file():
            raise FileNotFoundError(
                f"Required dataset file not found on Google Drive: {drive_file}. "
                "Place the exact unmodified train.json and validation.json in that folder and re-run this cell. "
                "No other dataset will be substituted."
            )
        drive_digest = sha256_of_file(drive_file)
        expected_digest = EXPECTED_SPLIT_SPECS[split_name]["sha256"]
        if drive_digest != expected_digest:
            raise ValueError(
                f"SHA-256 mismatch for Drive file '{drive_file}'!\n"
                f"  Expected : {expected_digest}\n"
                f"  Actual   : {drive_digest}\n"
                "Refusing to load; replace it with the exact unmodified split file."
            )
    shutil.copy2(DRIVE_INPUTS_DIR / "train.json", train_dst)
    shutil.copy2(DRIVE_INPUTS_DIR / "validation.json", val_dst)
    print(f"Loaded exact 'train.json' and 'validation.json' from {DRIVE_INPUTS_DIR} (Drive SHA-256 verified; no upload requested)")

# Option B (Drive NOT mounted only): direct browser upload of ONLY train.json and validation.json
if DRIVE_INPUTS_DIR is None and not (train_dst.exists() and val_dst.exists()):
    from google.colab import files
    missing_now = sorted(name for name in ALLOWED_UPLOAD_FILENAMES if not (DATA_DIR / name).exists())
    print(
        f"Missing required split file(s): {missing_now}\n"
        "Use Ctrl+Click (Windows/Linux) or Cmd+Click (macOS) to select ONLY "
        "'train.json' and 'validation.json' from C:\\slm-customer-support\\data\\splits\\:"
    )
    uploaded = files.upload()
    for fname, raw_bytes in uploaded.items():
        clean_name = Path(fname).name
        # Remove Colab's default copy in /content working directory to keep workspace tidy
        Path(fname).unlink(missing_ok=True)
        if clean_name not in ALLOWED_UPLOAD_FILENAMES:
            raise PermissionError(
                f"Quarantine / Upload Violation: Unexpected file '{clean_name}' rejected and deleted. "
                f"Only exact filenames {sorted(ALLOWED_UPLOAD_FILENAMES)} are permitted."
            )
        (DATA_DIR / clean_name).write_bytes(raw_bytes)

# Reject missing files explicitly
missing_after = sorted(name for name in ALLOWED_UPLOAD_FILENAMES if not (DATA_DIR / name).is_file())
if missing_after:
    raise FileNotFoundError(
        f"Required dataset file(s) still missing: {missing_after}. "
        "Re-run this cell and upload the missing file(s) before proceeding."
    )


def load_and_verify_split(file_path: Path) -> tuple[list[dict], str]:
    if file_path.name not in EXPECTED_SPLIT_SPECS:
        raise PermissionError(f"Unauthorized dataset file: {file_path.name}")
    spec = EXPECTED_SPLIT_SPECS[file_path.name]
    digest = sha256_of_file(file_path)
    if digest != spec["sha256"]:
        raise ValueError(
            f"SHA-256 mismatch for '{file_path.name}'!\n"
            f"  Expected : {spec['sha256']}\n"
            f"  Actual   : {digest}\n"
            "Ensure you uploaded the exact unmodified split file from C:\\slm-customer-support\\data\\splits\\."
        )

    records = json.loads(file_path.read_text(encoding="utf-8"))
    if not isinstance(records, list) or len(records) != spec["count"]:
        raise ValueError(
            f"Expected {spec['count']} records in {file_path.name}, found {len(records) if isinstance(records, list) else type(records)}."
        )

    seen_ids = set()
    for idx, rec in enumerate(records):
        missing = REQUIRED_RECORD_KEYS - set(rec.keys())
        if missing:
            raise ValueError(f"Record {idx} in {file_path.name} missing keys: {missing}")
        if not str(rec["question"]).strip() or not str(rec["expected_answer"]).strip():
            raise ValueError(f"Record {rec['id']} in {file_path.name} has empty question or answer.")
        if rec["id"] in seen_ids:
            raise ValueError(f"Duplicate ID {rec['id']} in {file_path.name}.")
        seen_ids.add(rec["id"])
        for field in ("question", "context", "expected_answer"):
            for tok in FORBIDDEN_CHATML_TOKENS:
                if tok in str(rec[field]):
                    raise ValueError(f"Special token {tok} found inside {file_path.name} record {rec['id']}.")
    return records, digest


train_records, TRAIN_SHA256 = load_and_verify_split(train_dst)
val_records, VAL_SHA256 = load_and_verify_split(val_dst)

overlap_ids = {r["id"] for r in train_records} & {r["id"] for r in val_records}
if overlap_ids:
    raise ValueError(f"ID leakage between train and validation splits: {overlap_ids}")

print(f"Verified train.json      : {len(train_records)} records | SHA-256 MATCH ({TRAIN_SHA256})")
print(f"Verified validation.json : {len(val_records)} records | SHA-256 MATCH ({VAL_SHA256})")
print("Cross-split ID overlap   : 0 records")

In [ ]:
print('CHECKPOINT: Section 6 completed successfully.')


---
### 7. Mandatory Pre-Training Runtime Compatibility Gate (Stops Before Model Download on Any Failure)
Verifies **all** hardware, CUDA, Python, package, mixed-precision, and `bitsandbytes` 4-bit NF4 requirements **before** downloading any Hugging Face model weights:

- Python version (`3.10–3.12`), CUDA runtime availability, GPU model, compute capability (`>= 7.0`), and **available free VRAM** (`>= 8.0 GiB`).
- Exact compatibility of `torch`, `transformers==4.48.3`, `peft==0.14.0`, `accelerate==1.3.0`, `safetensors==0.5.2`, and `bitsandbytes==0.45.3` (never assuming Colab's preinstalled packages are compatible).
- **NVIDIA T4 Strict Policy:** Requires `FP16` (`USE_FP16=True, USE_BF16=False`), **rejects any `BF16` configuration**, and executes a live in-memory `bnb.nn.Linear4bit(..., quant_type="nf4")` CUDA probe on GPU `0` to confirm `bitsandbytes` initializes cleanly.

In [ ]:
# ------------------------------------------------------------------------------
# Section 7 prerequisites: idempotently (re)bind imports and Section 2 configuration
# if the kernel state lacks them (e.g., Sections 2/4 were skipped or a restart occurred).
# Nothing is downloaded or installed here; values are identical to Sections 2 and 4.
# ------------------------------------------------------------------------------
import os
import sys

from packaging.version import Version

RUNTIME_COMPATIBILITY_GATE_PASSED = False  # Becomes True ONLY after every check below succeeds

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

try:
    import torch
    import accelerate
    import bitsandbytes as bnb
    import peft
    import safetensors
    import transformers
except ImportError as exc:
    raise RuntimeError(
        f"Required pinned package is not importable ({exc}). "
        "Run Section 4 (and restart the runtime if prompted), then re-run Sections 2-7 in order."
    ) from exc

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected! In Google Colab, select Runtime -> Change runtime type -> T4 GPU."
    )

# Original approved Section 2 thresholds (only defined when missing; never overridden)
if "MIN_VRAM_GB" not in globals():
    MIN_VRAM_GB = 8.0
if "MIN_COMPUTE_CAPABILITY" not in globals():
    MIN_COMPUTE_CAPABILITY = 7.0

# Original approved Section 2 hardware/precision policy (T4 -> FP16 strictly; Ampere+ -> BF16)
_SECTION2_NAMES = (
    "gpu_name", "gpu_vram_gb", "cc_tuple", "cc_float", "IS_T4_GPU",
    "USE_FP16", "USE_BF16", "COMPUTE_DTYPE", "PRECISION_LABEL",
)
if not all(_name in globals() for _name in _SECTION2_NAMES):
    _props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = _props.total_memory / (1024 ** 3)
    cc_tuple = (_props.major, _props.minor)
    cc_float = float(f"{_props.major}.{_props.minor}")
    IS_T4_GPU = ("T4" in gpu_name.upper()) or (cc_tuple == (7, 5))
    USE_BF16 = bool((not IS_T4_GPU) and _props.major >= 8 and bool(torch.cuda.is_bf16_supported()))
    USE_FP16 = not USE_BF16
    COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    PRECISION_LABEL = "bfloat16" if USE_BF16 else "float16"
    print("Restored missing Section 2 hardware/precision variables from the approved policy.")

# Consistency of precision variables (guards against conflicting definitions)
if USE_FP16 == USE_BF16:
    raise RuntimeError(f"Conflicting precision flags: USE_FP16={USE_FP16}, USE_BF16={USE_BF16}.")
if COMPUTE_DTYPE != (torch.bfloat16 if USE_BF16 else torch.float16) or PRECISION_LABEL != (
    "bfloat16" if USE_BF16 else "float16"
):
    raise RuntimeError(
        f"Conflicting precision configuration: COMPUTE_DTYPE={COMPUTE_DTYPE}, PRECISION_LABEL={PRECISION_LABEL}."
    )


def run_pretraining_compatibility_gate() -> dict:
    # 1. Python version check
    py_tuple = (sys.version_info.major, sys.version_info.minor)
    if not ((3, 10) <= py_tuple <= (3, 12)):
        raise RuntimeError(f"Incompatible Python {sys.version.split()[0]}; expected 3.10–3.12.")

    # 2. GPU & CUDA availability check
    if not torch.cuda.is_available() or not torch.version.cuda:
        raise RuntimeError("CUDA GPU unavailable or PyTorch lacks CUDA support.")

    live_gpu_name = torch.cuda.get_device_name(0)
    live_props = torch.cuda.get_device_properties(0)
    live_cc = (live_props.major, live_props.minor)
    live_cc_float = float(f"{live_props.major}.{live_props.minor}")
    live_free_bytes, live_total_bytes = torch.cuda.mem_get_info(0)
    live_free_gib = live_free_bytes / (1024 ** 3)
    live_total_gib = live_total_bytes / (1024 ** 3)

    if live_cc_float < MIN_COMPUTE_CAPABILITY:
        raise RuntimeError(f"GPU '{live_gpu_name}' compute capability {live_cc_float} < {MIN_COMPUTE_CAPABILITY}.")
    if live_free_gib < MIN_VRAM_GB:
        raise RuntimeError(
            f"Insufficient free GPU VRAM on '{live_gpu_name}': {live_free_gib:.2f} GiB available "
            f"(minimum required: {MIN_VRAM_GB:.2f} GiB). Restart the runtime to clear GPU memory."
        )

    # 3. Package version verification (do not assume Colab preinstalled packages match)
    torch_base_ver = Version(torch.__version__.split("+")[0])
    if not (Version("2.4.0") <= torch_base_ver < Version("2.8.0")):
        raise RuntimeError(f"Unsupported PyTorch version {torch.__version__}; expected >=2.4.0,<2.8.0.")

    expected_exact = {
        "transformers": (transformers.__version__, "4.48.3"),
        "peft": (peft.__version__, "0.14.0"),
        "accelerate": (accelerate.__version__, "1.3.0"),
        "safetensors": (safetensors.__version__, "0.5.2"),
        "bitsandbytes": (bnb.__version__, "0.45.3"),
    }
    for pkg_label, (actual_v, req_v) in expected_exact.items():
        if Version(actual_v.split("+")[0]) != Version(req_v):
            raise RuntimeError(
                f"Package '{pkg_label}' version mismatch: found {actual_v}, required {req_v}. "
                "Re-run Section 4 (and restart runtime if prompted)."
            )

    # 4. FP16 / BF16 mutual exclusion & NVIDIA T4 strict FP16 enforcement
    if USE_FP16 == USE_BF16:
        raise RuntimeError(f"Invalid precision state: USE_FP16={USE_FP16}, USE_BF16={USE_BF16}.")

    live_is_t4 = ("T4" in live_gpu_name.upper()) or (live_cc == (7, 5))
    if live_is_t4:
        if USE_BF16 or not USE_FP16 or COMPUTE_DTYPE != torch.float16:
            raise RuntimeError(
                f"NVIDIA T4 ('{live_gpu_name}', cc={live_cc_float}) requires FP16 training; "
                "BF16 configuration is strictly rejected."
            )

    # 5. Live bitsandbytes CUDA initialization & 4-bit NF4 double-quantization functional test
    probe_layer = None
    probe_input = None
    try:
        probe_layer = bnb.nn.Linear4bit(
            64,
            64,
            bias=False,
            compute_dtype=COMPUTE_DTYPE,
            compress_statistics=True,
            quant_type="nf4",
        ).to("cuda:0")
        q_state = getattr(probe_layer.weight, "quant_state", None)
        if q_state is None or getattr(q_state, "quant_type", None) != "nf4":
            raise RuntimeError("bitsandbytes Linear4bit failed to initialize NF4 quantization state.")
        probe_input = torch.ones((2, 64), device="cuda:0", dtype=COMPUTE_DTYPE)
        with torch.no_grad():
            probe_output = probe_layer(probe_input)
        if not torch.isfinite(probe_output).all().item():
            raise RuntimeError("bitsandbytes 4-bit NF4 forward probe produced non-finite values.")
    except Exception as exc:
        raise RuntimeError(
            f"bitsandbytes CUDA / 4-bit NF4 verification failed on '{live_gpu_name}': {exc}. "
            "Execution halted safely BEFORE downloading any model checkpoint."
        ) from exc
    finally:
        del probe_layer, probe_input
        torch.cuda.empty_cache()

    return {
        "python": sys.version.split()[0],
        "gpu_model": live_gpu_name,
        "compute_capability": live_cc_float,
        "free_vram_gib": round(live_free_gib, 2),
        "total_vram_gib": round(live_total_gib, 2),
        "cuda": torch.version.cuda,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "bitsandbytes": bnb.__version__,
        "precision": PRECISION_LABEL,
        "bnb_4bit_nf4_verified": True,
    }


COMPATIBILITY_REPORT = run_pretraining_compatibility_gate()
RUNTIME_COMPATIBILITY_GATE_PASSED = True

print("MANDATORY PRE-TRAINING COMPATIBILITY GATE: PASSED")
for k, v in COMPATIBILITY_REPORT.items():
    print(f"  - {k:24s}: {v}")

---
### Preflight complete
If the cell below prints `PREFLIGHT COMPLETE`, return to the main notebook (Section 8 onward needs your separate approval). Nothing is executed beyond this point.

In [ ]:
# Final checkpoint (read-only): confirms the genuine Section 7 gate passed. Does not set any flag.
if globals().get("RUNTIME_COMPATIBILITY_GATE_PASSED", False) is not True:
    raise RuntimeError("Preflight FAILED: Section 7 compatibility gate did not pass.")
assert USE_FP16 and not USE_BF16 and PRECISION_LABEL == "float16", "T4 FP16 configuration not active."
assert len(train_records) == 250 and len(val_records) == 50
print("PREFLIGHT COMPLETE: Sections 2-7 passed. STOP HERE - do not run Section 8 from this notebook.")


---
### Diagnostic Summary (manual)
Run the next cell after any failure to list recorded events. It is read-only.

In [ ]:
# Diagnostic summary (read-only). Run manually after any failure; it lists all recorded events.
import json as _j
from pathlib import Path as _Pp
_ev = sorted((_DIAG_LOCAL_DIR / "events").glob("*.json"))
print(f"{len(_ev)} diagnostic event(s) in {_DIAG_LOCAL_DIR / 'events'}")
for _f in _ev:
    _e = _j.loads(_f.read_text(encoding="utf-8"))
    print(f"- {_e['timestamp_utc']} | {_e['section']} ({_e['cell_id']}) | {_e['status']}"
          + (f" | {_e['exception_type']}: {_e['exception_message'][:160]}" if _e['status'] == 'FAILED' else ""))
print("Drive sync status:", _diag_sync_to_drive())
